# 3.审批模式

In [ ]:
from langchain_core.messages import HumanMessage
from langchain.chat_models import init_chat_model
from typing import TypedDict, Literal

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import Command
from langgraph.types import interrupt
from dotenv import load_dotenv

load_dotenv(override=True)

model = init_chat_model("deepseek-flash", )


# res = model.invoke([HumanMessage("你好"),])
# rprint(res)


# 声明状态
class State(TypedDict):
    topic: str
    poem: str
    is_approved: bool


# 声明节点
def approved_node(state: State) -> Command[Literal["llm_node", "default_node"]]:
    is_approved = interrupt("是否同意调用模型？")
    goto = "llm_node" if is_approved else "default_node"
    return Command(goto=goto, update={"is_approved": is_approved})


def llm_node(state: State) -> dict:
    topic = state["topic"]
    poem = model.invoke([HumanMessage(f"写一首关于{topic}的短诗")]).content
    return {"poem": poem}


def default_node(state: State) -> dict:
    return {"poem": "请求被拒绝"}


# 构建图结构
builder = StateGraph(state_schema=State)
builder.add_node("approved_node", approved_node)
builder.add_node("llm_node", llm_node)
builder.add_node("default_node", default_node)
builder.add_edge(START, "approved_node")
builder.add_edge("llm_node", END)
builder.add_edge("default_node", END)

# 创建检查点
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer)
config = {"configurable": {"thread_id": "1"}}

# 执行 -> 触发中断
res = graph.invoke({"topic": "月亮"}, config=config)
print(res)

In [ ]:
# 用户审批 -> 恢复执行
ask = res["__interrupt__"][0].value
approve_res = input(ask).strip().lower() in ("y", "yes", "是", "1")

resume_res = graph.invoke(Command(resume=approve_res), config=config)
print(resume_res)
